# e-waste classification -- 18 classes
**SDG 12.4 | Predictive Analysis Project**  
transfer learning comparison: ResNet18 vs ResNet50 vs EfficientNet-B0 vs ViT-B/16

## environment

In [1]:
import torch, sys
print(f"python  : {sys.version.split()[0]}")
print(f"pytorch : {torch.__version__}")
print(f"cuda    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu     : {torch.cuda.get_device_name(0)}")
    print(f"vram    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} gb")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device  : {device}")


python  : 3.12.10
pytorch : 2.11.0+cu126
cuda    : True
gpu     : NVIDIA GeForce RTX 3050 6GB Laptop GPU
vram    : 6.4 gb
device  : cuda


## imports

In [2]:
import os, time, json, warnings, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from copy import deepcopy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, accuracy_score, precision_score, recall_score)
from tqdm import tqdm

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)
torch.backends.cudnn.benchmark = True


## configuration -- 18 classes

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# paths
DATA_DIR   = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "models" / "classification"
GRAPHS_DIR = OUTPUT_DIR / "graphs"
MODELS_DIR = OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GRAPHS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

CONFIG = {
    "img_size"       : 224,
    "batch_size"     : 32,
    "num_epochs"     : 30,
    "lr"             : 1e-4,
    "weight_decay"   : 1e-4,
    "num_workers"    : 4,
    "patience"       : 7,
    "unfreeze_epoch" : 5,
    "embed_batch"    : 64,
}

# 18 classes -- final structure after all merges
# high (8): refrigerant-containing appliances + high-toxicity devices
# medium (4): large appliances with recoverable materials
# low (6): passive components + plastic-dominant devices

HAZARD_MAP = {
    "Battery"           : "HIGH",     # lithium / cadmium -- Basel Annex I
    "PCB"               : "HIGH",     # lead solder / brominated flame retardants
    "Mobile"            : "HIGH",     # lithium + rare earth metals
    "Television"        : "HIGH",     # crt lead glass / mercury
    "Laptop"            : "HIGH",     # lithium + lead solder
    "light bulbs"       : "HIGH",     # mercury (CFL) -- Basel Annex I
    "Refrigerator"      : "HIGH",     # CFC / HCFC refrigerants -- ozone-depleting
    "Air-Conditioner"   : "HIGH",     # HCFC refrigerants -- Basel Convention
    "Microwave"         : "MEDIUM",   # magnetron / steel -- recoverable
    "Washing Machine"   : "MEDIUM",   # steel / copper motor -- recoverable
    "Printer"           : "MEDIUM",   # toner / circuit -- moderate hazard
    "Microchip-IC"      : "MEDIUM",   # silicon / gold traces -- recoverable
    "Keyboard"          : "LOW",      # ABS plastic -- recyclable
    "Mouse"             : "LOW",      # ABS plastic -- recyclable
    "Resistor"          : "LOW",      # ceramic / carbon -- inert
    "transistor"        : "LOW",      # silicon / germanium -- recoverable
    "heat-sink"         : "LOW",      # aluminium -- high value recovery
    "Passive-Component" : "LOW",      # Capacitor + LED + semiconductor-diode merged
}

DISPOSAL_MAP = {
    "Battery"           : "hazardous waste facility -- lithium/cadmium recovery",
    "PCB"               : "certified e-waste recycler -- gold/copper extraction",
    "Mobile"            : "certified e-waste recycler -- rare earth recovery",
    "Television"        : "hazardous waste facility -- crt lead/mercury handling",
    "Laptop"            : "certified e-waste recycler -- battery + rare earth",
    "light bulbs"       : "hazardous waste facility -- mercury containment",
    "Refrigerator"      : "certified refrigerant recovery facility -- CFC/HCFC extraction before dismantling",
    "Air-Conditioner"   : "certified refrigerant recovery facility -- HCFC extraction before dismantling",
    "Microwave"         : "metal recycler -- steel/copper/magnetron recovery",
    "Washing Machine"   : "metal recycler -- steel/motor/copper recovery",
    "Printer"           : "e-waste recycler -- toner/circuit recovery",
    "Microchip-IC"      : "e-waste recycler -- silicon/gold recovery",
    "Keyboard"          : "plastic recycler -- abs plastic stream",
    "Mouse"             : "plastic recycler -- abs plastic stream",
    "Resistor"          : "component recycler -- ceramic/carbon recovery",
    "transistor"        : "component recycler -- semiconductor recovery",
    "heat-sink"         : "metal recycler -- aluminium recovery",
    "Passive-Component" : "component recycler -- semiconductor and aluminium recovery",
}

MATERIAL_MAP = {
    "Battery"           : "lithium / cadmium / lead acid",
    "PCB"               : "fr4 composite / lead solder / gold / copper",
    "Mobile"            : "lithium / rare earth metals / glass",
    "Television"        : "lead glass / mercury / plastic composite",
    "Laptop"            : "lithium / rare earth / aluminium / lead solder",
    "light bulbs"       : "glass / mercury / argon / tungsten",
    "Refrigerator"      : "steel / CFC-R12 or HCFC-R22 refrigerant / polyurethane foam / copper",
    "Air-Conditioner"   : "aluminium / HCFC-R22 or HFC-R410A refrigerant / copper / steel",
    "Microwave"         : "steel / copper / magnetron / ceramic",
    "Washing Machine"   : "steel / copper motor / rubber / plastic",
    "Printer"           : "abs plastic / toner / copper circuit / lead",
    "Microchip-IC"      : "silicon / gold bond wires / lead solder / ceramic",
    "Keyboard"          : "abs plastic / rubber / copper traces",
    "Mouse"             : "abs plastic / optical sensor / copper",
    "Resistor"          : "carbon / ceramic / metal film",
    "transistor"        : "silicon / germanium / plastic / lead",
    "heat-sink"         : "aluminium / copper / thermal compound",
    "Passive-Component" : "silicon / aluminium / gallium nitride / ceramic",
}

HAZARD_COLORS = {"HIGH": "#d7191c", "MEDIUM": "#fdae61", "LOW": "#1a9641"}
HAZARD_INT    = {"HIGH": 0, "MEDIUM": 1, "LOW": 2}

print("configuration loaded -- 18 classes")
for h in ["HIGH", "MEDIUM", "LOW"]:
    cls_list = [c for c, v in HAZARD_MAP.items() if v == h]
    print(f"  {h:<8}: {len(cls_list)} classes -- {cls_list}")


# override output for classification-specific results
CLS_OUTPUT = PROJECT_ROOT / "models" / "classification"
CLS_GRAPHS = CLS_OUTPUT / "graphs"
CLS_OUTPUT.mkdir(parents=True, exist_ok=True)
CLS_GRAPHS.mkdir(exist_ok=True)


configuration loaded -- 18 classes
  HIGH    : 8 classes -- ['Battery', 'PCB', 'Mobile', 'Television', 'Laptop', 'light bulbs', 'Refrigerator', 'Air-Conditioner']
  MEDIUM  : 4 classes -- ['Microwave', 'Washing Machine', 'Printer', 'Microchip-IC']
  LOW     : 6 classes -- ['Keyboard', 'Mouse', 'Resistor', 'transistor', 'heat-sink', 'Passive-Component']


## data pipeline

In [4]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((CONFIG["img_size"] + 32, CONFIG["img_size"] + 32)),
    transforms.RandomCrop(CONFIG["img_size"]),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
])
eval_transform = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = datasets.ImageFolder(DATA_DIR / "train", transform=train_transform)
val_ds   = datasets.ImageFolder(DATA_DIR / "val",   transform=eval_transform)
test_ds  = datasets.ImageFolder(DATA_DIR / "test",  transform=eval_transform)

CLASS_NAMES = train_ds.classes
NUM_CLASSES = len(CLASS_NAMES)

assert NUM_CLASSES == 18, f"expected 18 classes, found {NUM_CLASSES} -- check your dataset"
print(f"classes : {NUM_CLASSES}")
print(f"train   : {len(train_ds)}")
print(f"val     : {len(val_ds)}")
print(f"test    : {len(test_ds)}")

missing_hazard = [c for c in CLASS_NAMES if c not in HAZARD_MAP]
if missing_hazard:
    print(f"warning -- no hazard mapping for: {missing_hazard}")
else:
    print("all 18 classes have hazard mapping")


classes : 18
train   : 23960
val     : 1800
test    : 1800
all 18 classes have hazard mapping


## weighted sampler

In [5]:
class_counts   = np.bincount([s[1] for s in train_ds.samples])
class_weights  = 1.0 / class_counts
sample_weights = np.array([class_weights[s[1]] for s in train_ds.samples])

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"],
                          sampler=sampler, num_workers=CONFIG["num_workers"],
                          pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=CONFIG["num_workers"],
                          pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=CONFIG["num_workers"],
                          pin_memory=True)
print(f"train batches : {len(train_loader)}")
print(f"val batches   : {len(val_loader)}")


train batches : 749
val batches   : 57


## model factory

In [6]:
def build_model(arch, num_classes):
    if arch == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        in_dim = m.fc.in_features
        m.fc = nn.Sequential(
            nn.Linear(in_dim, 512), nn.BatchNorm1d(512),
            nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(256, num_classes))
    elif arch == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        in_dim = m.fc.in_features
        m.fc = nn.Sequential(
            nn.Linear(in_dim, 512), nn.BatchNorm1d(512),
            nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(256, num_classes))
    elif arch == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_dim = m.classifier[1].in_features
        m.classifier = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(in_dim, 256),
            nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(256, num_classes))
    elif arch == "vit_b16":
        m = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
        in_dim = m.heads.head.in_features
        m.heads.head = nn.Sequential(
            nn.Linear(in_dim, 512), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(512, num_classes))
    else:
        raise ValueError(f"unknown arch: {arch}")

    for name, param in m.named_parameters():
        if not any(k in name for k in ["fc", "classifier", "heads"]):
            param.requires_grad = False
    return m


def unfreeze(model, arch):
    keys = {
        "resnet18"       : ["layer3", "layer4", "fc"],
        "resnet50"       : ["layer3", "layer4", "fc"],
        "efficientnet_b0": ["features.6", "features.7", "features.8", "classifier"],
        "vit_b16"        : ["encoder.layers.10", "encoder.layers.11", "heads"],
    }
    for name, param in model.named_parameters():
        if any(k in name for k in keys.get(arch, [])):
            param.requires_grad = True
    return model


print("model factory ready")


model factory ready


## training engine

In [7]:
def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    loss_sum = correct = total = 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device, non_blocking=True), lbls.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            out  = model(imgs)
            loss = criterion(out, lbls)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * imgs.size(0)
        _, p = torch.max(out, 1)
        correct += (p == lbls).sum().item()
        total   += lbls.size(0)
    return loss_sum / total, correct / total


def eval_epoch(model, loader, criterion):
    model.eval()
    loss_sum = correct = total = 0
    preds, labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device, non_blocking=True), lbls.to(device, non_blocking=True)
            with autocast():
                out  = model(imgs)
                loss = criterion(out, lbls)
            loss_sum += loss.item() * imgs.size(0)
            _, p = torch.max(out, 1)
            correct += (p == lbls).sum().item()
            total   += lbls.size(0)
            preds.extend(p.cpu().numpy())
            labels.extend(lbls.cpu().numpy())
    return loss_sum / total, correct / total, preds, labels


def train_model(arch, save_dir):
    print(f"\ntraining {arch} -- 18 classes")
    print("-" * 50)
    model     = build_model(arch, NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CONFIG["num_epochs"], eta_min=1e-6)
    scaler = GradScaler()
    history = {"train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[], "lr":[]}
    best_val = 0.0
    best_w   = None
    pat_ctr  = 0

    for epoch in range(1, CONFIG["num_epochs"] + 1):
        t0 = time.time()
        if epoch == CONFIG["unfreeze_epoch"]:
            model = unfreeze(model, arch)
            optimizer = optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=CONFIG["lr"] * 0.1, weight_decay=CONFIG["weight_decay"])
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=CONFIG["num_epochs"] - epoch, eta_min=1e-7)
            print(f"  [{epoch}] backbone unfrozen")

        tl, ta = train_epoch(model, train_loader, criterion, optimizer, scaler)
        vl, va, _, _ = eval_epoch(model, val_loader, criterion)
        scheduler.step()
        lr = optimizer.param_groups[0]["lr"]
        for k, v in zip(["train_loss","train_acc","val_loss","val_acc","lr"],
                         [tl, ta, vl, va, lr]):
            history[k].append(v)

        print(f"  {epoch:02d}/{CONFIG['num_epochs']} | loss {tl:.4f}/{vl:.4f} | "
              f"acc {ta:.4f}/{va:.4f} | lr {lr:.2e} | {time.time()-t0:.1f}s")

        if va > best_val:
            best_val = va
            best_w   = deepcopy(model.state_dict())
            torch.save(model.state_dict(), save_dir / f"{arch}_best.pth")
            pat_ctr  = 0
        else:
            pat_ctr += 1
            if pat_ctr >= CONFIG["patience"]:
                print(f"  early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_w)
    print(f"  best val accuracy: {best_val:.4f}")
    return model, history


print("training engine ready")


training engine ready


## train all architectures

In [8]:
ARCHS = ["resnet18", "resnet50", "efficientnet_b0", "vit_b16"]
dl_models = {}
dl_histories = {}

for arch in ARCHS:
    d = MODELS_DIR / arch
    d.mkdir(exist_ok=True)
    model, history = train_model(arch, d)
    dl_models[arch]    = model
    dl_histories[arch] = history
    torch.cuda.empty_cache()

print("all models trained")



training resnet18 -- 18 classes
--------------------------------------------------
  01/30 | loss 1.9570/1.2260 | acc 0.5192/0.7756 | lr 9.97e-05 | 91.3s
  02/30 | loss 1.4326/1.1407 | acc 0.6902/0.8111 | lr 9.89e-05 | 95.3s
  03/30 | loss 1.3431/1.1250 | acc 0.7243/0.8111 | lr 9.76e-05 | 107.7s
  04/30 | loss 1.2842/1.0832 | acc 0.7472/0.8256 | lr 9.57e-05 | 115.3s
  [5] backbone unfrozen
  05/30 | loss 1.1891/0.9947 | acc 0.7863/0.8706 | lr 9.96e-06 | 129.6s
  06/30 | loss 1.0817/0.9545 | acc 0.8341/0.8850 | lr 9.84e-06 | 97.6s
  07/30 | loss 1.0241/0.9166 | acc 0.8582/0.9000 | lr 9.65e-06 | 85.2s
  08/30 | loss 0.9843/0.8938 | acc 0.8761/0.9067 | lr 9.39e-06 | 87.9s
  09/30 | loss 0.9523/0.8712 | acc 0.8873/0.9178 | lr 9.05e-06 | 85.9s
  10/30 | loss 0.9271/0.8640 | acc 0.8980/0.9139 | lr 8.66e-06 | 89.3s
  11/30 | loss 0.9081/0.8601 | acc 0.9069/0.9194 | lr 8.21e-06 | 88.3s
  12/30 | loss 0.8841/0.8512 | acc 0.9159/0.9211 | lr 7.70e-06 | 86.6s
  13/30 | loss 0.8701/0.8396 | acc 0.

## evaluate on test set

In [9]:
dl_results = {}
criterion  = nn.CrossEntropyLoss()

print(f"{'model':<20} {'accuracy':>10} {'macro_f1':>10} {'weighted_f1':>12}")
print("-" * 56)

for arch in ARCHS:
    _, acc, preds, labels = eval_epoch(dl_models[arch], test_loader, criterion)
    mf1 = f1_score(labels, preds, average="macro")
    wf1 = f1_score(labels, preds, average="weighted")
    dl_results[arch] = {
        "accuracy": round(acc, 4), "macro_f1": round(mf1, 4),
        "weighted_f1": round(wf1, 4),
        "macro_precision": round(precision_score(labels, preds, average="macro"), 4),
        "macro_recall": round(recall_score(labels, preds, average="macro"), 4),
        "preds": preds, "labels": labels,
    }
    print(f"  {arch:<18} {acc:>10.4f} {mf1:>10.4f} {wf1:>12.4f}")

best_dl = max(ARCHS, key=lambda a: dl_results[a]["macro_f1"])
print(f"\nbest: {best_dl}")
with open(CLS_OUTPUT / "best_model.json", "w") as f:
    json.dump({"best_arch": best_dl}, f, indent=2)


model                  accuracy   macro_f1  weighted_f1
--------------------------------------------------------
  resnet18               0.9450     0.9445       0.9445
  resnet50               0.9583     0.9581       0.9581
  efficientnet_b0        0.9411     0.9407       0.9407
  vit_b16                0.9344     0.9339       0.9339

best: resnet50


## confusion matrices

In [10]:
fig, axes = plt.subplots(1, 4, figsize=(36, 10))
fig.suptitle("confusion matrices -- 18-class e-waste test set", fontsize=14, fontweight="bold")

for idx, arch in enumerate(ARCHS):
    cm = confusion_matrix(dl_results[arch]["labels"], dl_results[arch]["preds"])
    cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_norm, ax=axes[idx], annot=True, fmt=".2f", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=0.3, vmin=0, vmax=1)
    axes[idx].set_title(
        f"{arch}\nacc={dl_results[arch]['accuracy']:.4f}  f1={dl_results[arch]['macro_f1']:.4f}",
        fontsize=10)
    axes[idx].set_ylabel("true label", fontsize=9)
    axes[idx].set_xlabel("predicted label", fontsize=9)
    axes[idx].tick_params(axis="x", rotation=45, labelsize=7)
    axes[idx].tick_params(axis="y", rotation=0, labelsize=7)

plt.tight_layout()
plt.savefig(CLS_GRAPHS / "confusion_matrices_18cls.png", dpi=150, bbox_inches="tight")
plt.close()
print("confusion matrices saved")


confusion matrices saved


## training curves

In [11]:
fig, axes = plt.subplots(2, 4, figsize=(24, 10))
fig.suptitle("training curves -- 18-class e-waste", fontsize=14, fontweight="bold")

for idx, arch in enumerate(ARCHS):
    h = dl_histories[arch]
    e = range(1, len(h["train_loss"]) + 1)
    axes[0, idx].plot(e, h["train_loss"], color="#2c7bb6", linewidth=1.5, label="train")
    axes[0, idx].plot(e, h["val_loss"],   color="#d7191c", linewidth=1.5, label="val")
    axes[0, idx].set_title(f"{arch} -- loss", fontsize=10)
    axes[0, idx].set_xlabel("epoch"); axes[0, idx].legend(fontsize=8); axes[0, idx].grid(alpha=0.3)
    axes[1, idx].plot(e, h["train_acc"], color="#2c7bb6", linewidth=1.5, label="train")
    axes[1, idx].plot(e, h["val_acc"],   color="#d7191c", linewidth=1.5, label="val")
    axes[1, idx].set_title(f"{arch} -- accuracy", fontsize=10)
    axes[1, idx].set_xlabel("epoch"); axes[1, idx].legend(fontsize=8)
    axes[1, idx].grid(alpha=0.3); axes[1, idx].set_ylim(0, 1)

plt.tight_layout()
plt.savefig(CLS_GRAPHS / "training_curves_18cls.png", dpi=150, bbox_inches="tight")
plt.close()
print("training curves saved")

with open(CLS_OUTPUT / "dl_results.json", "w") as f:
    json.dump({k: {m: v for m, v in r.items() if m not in ["preds","labels"]}
               for k, r in dl_results.items()}, f, indent=2)
print("results saved")


training curves saved
results saved
